In [1]:
# ========================
# Week 8 — Phase 1 Capstone
# SecureChat — Encrypted Chat Application
# ========================

from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.fernet import Fernet
import hashlib
import os
from datetime import datetime

def generate_rsa_keypair(name):
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )
    public_key = private_key.public_key()
    
    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    public_pem = public_key.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    )
    
    os.makedirs('secure_chat_keys', exist_ok=True)
    with open(f'secure_chat_keys/{name}_private.pem', 'wb') as f:
        f.write(private_pem)
    with open(f'secure_chat_keys/{name}_public.pem', 'wb') as f:
        f.write(public_pem)
    
    print(f"✅ {name} key pair generated!")
    return private_key, public_key

# Generate keys for both users
print("🔐 SecureChat — Key Generation\n")
alice_private, alice_public = generate_rsa_keypair("alice")
bob_private,   bob_public   = generate_rsa_keypair("bob")
print(f"\n✅ Keys saved to secure_chat_keys/")
print(f"✅ Both users ready!")

🔐 SecureChat — Key Generation

✅ alice key pair generated!
✅ bob key pair generated!

✅ Keys saved to secure_chat_keys/
✅ Both users ready!


In [2]:
# Create proper project folder
mkdir -p ~/quantyber/projects/secure_chat

# Move keys to correct location
mv ~/quantyber/secure_chat_keys ~/quantyber/projects/secure_chat/

# Verify
ls ~/quantyber/projects/secure_chat/secure_chat_keys/

SyntaxError: invalid syntax (2649017005.py, line 2)

In [3]:
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import serialization
import json
import base64
import hashlib
from datetime import datetime

def encrypt_message(sender_private_key, receiver_public_key, message, sender_name):
    # Step 1 — Generate session key
    session_key = Fernet.generate_key()
    cipher = Fernet(session_key)
    
    # Step 2 — Encrypt message with AES
    encrypted_message = cipher.encrypt(message.encode())
    
    # Step 3 — Encrypt session key with RSA
    encrypted_session_key = receiver_public_key.encrypt(
        session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    
    # Step 4 — Sign message hash
    message_hash = hashlib.sha256(message.encode()).digest()
    signature = sender_private_key.sign(
        message_hash,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    
    # Package everything
    package = {
        "sender":            sender_name,
        "timestamp":         datetime.now().isoformat(),
        "encrypted_message": base64.b64encode(encrypted_message).decode(),
        "encrypted_key":     base64.b64encode(encrypted_session_key).decode(),
        "signature":         base64.b64encode(signature).decode(),
        "message_hash":      hashlib.sha256(message.encode()).hexdigest()
    }
    
    return json.dumps(package)

# Test encryption
message = "Hello Bob! This is a quantum-safe secret message from Alice 🔐"

print("🔐 ENCRYPTING MESSAGE")
print(f"Original: {message}\n")

encrypted_package = encrypt_message(
    alice_private,
    bob_public,
    message,
    "Alice"
)

package_data = json.loads(encrypted_package)
print(f"Sender:    {package_data['sender']}")
print(f"Timestamp: {package_data['timestamp']}")
print(f"Encrypted: {package_data['encrypted_message'][:50]}...")
print(f"Enc Key:   {package_data['encrypted_key'][:50]}...")
print(f"Signature: {package_data['signature'][:50]}...")
print(f"Hash:      {package_data['message_hash']}")

🔐 ENCRYPTING MESSAGE
Original: Hello Bob! This is a quantum-safe secret message from Alice 🔐

Sender:    Alice
Timestamp: 2026-07-29T12:10:00.183152
Encrypted: Z0FBQUFBQnFhYUJBbkFoczBERjBrMk9iRnhFQmEyYjl4bDYtbX...
Enc Key:   fMs/MyfbCPj2OSqs5u0s5BZov4bIz3lFLkE0zpFrbtGRb/Y5q3...
Signature: mUT5pxpwDA1aKFrviFmwBu35Ry5DcLVEVBwsI3ZzovFJBRHKtV...
Hash:      7bc6bf239409750da215125b499dd4c6df5c21507ffee37b6a02e2656269b762


In [4]:
def decrypt_message(receiver_private_key, sender_public_key, encrypted_package):
    """
    Decrypt and verify a message:
    1. Decrypt session key with receiver's private key
    2. Decrypt message with session key
    3. Verify signature with sender's public key
    4. Verify message hash integrity
    """
    
    package = json.loads(encrypted_package)
    
    print(f"📨 MESSAGE RECEIVED")
    print(f"From:      {package['sender']}")
    print(f"Timestamp: {package['timestamp']}")
    
    # Step 1 — Decrypt session key with Bob's private key
    encrypted_session_key = base64.b64decode(package['encrypted_key'])
    session_key = receiver_private_key.decrypt(
        encrypted_session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    print(f"\n✅ Session key decrypted")
    
    # Step 2 — Decrypt message with session key
    cipher = Fernet(session_key)
    encrypted_message = base64.b64decode(package['encrypted_message'])
    decrypted_message = cipher.decrypt(encrypted_message).decode()
    print(f"✅ Message decrypted: {decrypted_message}")
    
    # Step 3 — Verify signature
    try:
        message_hash = hashlib.sha256(decrypted_message.encode()).digest()
        signature = base64.b64decode(package['signature'])
        sender_public_key.verify(
            signature,
            message_hash,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print(f"✅ Signature verified — message is authentic!")
    except Exception:
        print(f"❌ Signature verification FAILED — message tampered!")
        return None
    
    # Step 4 — Verify hash integrity
    computed_hash = hashlib.sha256(decrypted_message.encode()).hexdigest()
    if computed_hash == package['message_hash']:
        print(f"✅ Hash verified — message integrity confirmed!")
    else:
        print(f"❌ Hash mismatch — message corrupted!")
        return None
    
    return decrypted_message

# Bob decrypts Alice's message
print("\n" + "="*55)
result = decrypt_message(bob_private, alice_public, encrypted_package)

print(f"\n{'='*55}")
print(f"📨 FINAL DECRYPTED MESSAGE:")
print(f"{result}")


📨 MESSAGE RECEIVED
From:      Alice
Timestamp: 2026-07-29T12:10:00.183152

✅ Session key decrypted
✅ Message decrypted: Hello Bob! This is a quantum-safe secret message from Alice 🔐
✅ Signature verified — message is authentic!
✅ Hash verified — message integrity confirmed!

📨 FINAL DECRYPTED MESSAGE:
Hello Bob! This is a quantum-safe secret message from Alice 🔐


In [5]:
import json

print("🔴 TAMPER DETECTION TEST")
print("="*55)

# Simulate attacker modifying the message
tampered_package = json.loads(encrypted_package)
tampered_package['encrypted_message'] = "TAMPERED_MESSAGE_BY_ATTACKER"
tampered_package_str = json.dumps(tampered_package)

print("Attacker modified encrypted_message...")
print("Attempting to decrypt tampered package:\n")

try:
    result = decrypt_message(bob_private, alice_public, tampered_package_str)
except Exception as e:
    print(f"❌ Decryption failed: {type(e).__name__}")
    print(f"✅ Tamper detected — message rejected!")

print("\n" + "="*55)

# Simulate attacker modifying plaintext after decryption
print("\nSimulating replay attack with wrong signature...")
fake_package = json.loads(encrypted_package)
fake_package['sender'] = "Attacker pretending to be Alice"
fake_package_str = json.dumps(fake_package)

result = decrypt_message(bob_private, alice_public, fake_package_str)

🔴 TAMPER DETECTION TEST
Attacker modified encrypted_message...
Attempting to decrypt tampered package:

📨 MESSAGE RECEIVED
From:      Alice
Timestamp: 2026-07-29T12:10:00.183152

✅ Session key decrypted
❌ Decryption failed: Error
✅ Tamper detected — message rejected!


Simulating replay attack with wrong signature...
📨 MESSAGE RECEIVED
From:      Attacker pretending to be Alice
Timestamp: 2026-07-29T12:10:00.183152

✅ Session key decrypted
✅ Message decrypted: Hello Bob! This is a quantum-safe secret message from Alice 🔐
✅ Signature verified — message is authentic!
✅ Hash verified — message integrity confirmed!


In [6]:
def encrypt_message_v2(sender_private_key, receiver_public_key, 
                        message, sender_name):
    
    session_key = Fernet.generate_key()
    cipher = Fernet(session_key)
    encrypted_message = cipher.encrypt(message.encode())
    
    encrypted_session_key = receiver_public_key.encrypt(
        session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    
    # FIX — Sign message + sender + timestamp together
    timestamp = datetime.now().isoformat()
    signed_content = f"{sender_name}:{timestamp}:{message}"
    signed_hash = hashlib.sha256(signed_content.encode()).digest()
    
    signature = sender_private_key.sign(
        signed_hash,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    
    package = {
        "sender":            sender_name,
        "timestamp":         timestamp,
        "encrypted_message": base64.b64encode(encrypted_message).decode(),
        "encrypted_key":     base64.b64encode(encrypted_session_key).decode(),
        "signature":         base64.b64encode(signature).decode(),
        "message_hash":      hashlib.sha256(signed_content.encode()).hexdigest()
    }
    
    return json.dumps(package)

def decrypt_message_v2(receiver_private_key, sender_public_key,
                        encrypted_package, expected_sender):
    
    package = json.loads(encrypted_package)
    
    print(f"📨 MESSAGE RECEIVED")
    print(f"From:      {package['sender']}")
    print(f"Timestamp: {package['timestamp']}")
    
    # Verify sender matches expected
    if package['sender'] != expected_sender:
        print(f"❌ SENDER MISMATCH — Expected {expected_sender}, got {package['sender']}")
        return None
    
    # Decrypt session key
    encrypted_session_key = base64.b64decode(package['encrypted_key'])
    session_key = receiver_private_key.decrypt(
        encrypted_session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    print(f"\n✅ Session key decrypted")
    
    # Decrypt message
    cipher = Fernet(session_key)
    encrypted_message = base64.b64decode(package['encrypted_message'])
    decrypted_message = cipher.decrypt(encrypted_message).decode()
    print(f"✅ Message decrypted: {decrypted_message}")
    
    # Verify signature — now includes sender + timestamp
    signed_content = f"{package['sender']}:{package['timestamp']}:{decrypted_message}"
    signed_hash = hashlib.sha256(signed_content.encode()).digest()
    
    try:
        signature = base64.b64decode(package['signature'])
        sender_public_key.verify(
            signature,
            signed_hash,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print(f"✅ Signature verified — sender authenticated!")
    except Exception:
        print(f"❌ Signature FAILED — identity spoofing detected!")
        return None
    
    # Verify hash
    computed_hash = hashlib.sha256(signed_content.encode()).hexdigest()
    if computed_hash == package['message_hash']:
        print(f"✅ Hash verified — integrity confirmed!")
    else:
        print(f"❌ Hash mismatch — message corrupted!")
        return None
    
    return decrypted_message

# Test v2 — legitimate message
print("="*55)
print("✅ TEST 1 — Legitimate message")
print("="*55)
encrypted_v2 = encrypt_message_v2(
    alice_private, bob_public,
    "Hello Bob! Secure message v2 🔐", "Alice"
)
decrypt_message_v2(bob_private, alice_public, encrypted_v2, "Alice")

# Test v2 — replay attack attempt
print("\n" + "="*55)
print("🔴 TEST 2 — Replay attack attempt")
print("="*55)
tampered_v2 = json.loads(encrypted_v2)
tampered_v2['sender'] = "Attacker pretending to be Alice"
decrypt_message_v2(
    bob_private, alice_public,
    json.dumps(tampered_v2),
    "Alice"
)

✅ TEST 1 — Legitimate message
📨 MESSAGE RECEIVED
From:      Alice
Timestamp: 2026-07-29T12:14:38.881649

✅ Session key decrypted
✅ Message decrypted: Hello Bob! Secure message v2 🔐
✅ Signature verified — sender authenticated!
✅ Hash verified — integrity confirmed!

🔴 TEST 2 — Replay attack attempt
📨 MESSAGE RECEIVED
From:      Attacker pretending to be Alice
Timestamp: 2026-07-29T12:14:38.881649
❌ SENDER MISMATCH — Expected Alice, got Attacker pretending to be Alice


In [7]:
import json
import os
from datetime import datetime

class AuditLogger:
    def __init__(self, log_file="secure_chat_audit.log"):
        self.log_file = log_file
        self.logs = []
    
    def log_event(self, event_type, sender, receiver, 
                  status, details=""):
        event = {
            "timestamp": datetime.now().isoformat(),
            "event":     event_type,
            "sender":    sender,
            "receiver":  receiver,
            "status":    status,
            "details":   details
        }
        self.logs.append(event)
        
        # Write to file
        with open(self.log_file, 'a') as f:
            f.write(json.dumps(event) + '\n')
        
        # Print to console
        status_icon = "✅" if status == "SUCCESS" else "❌"
        print(f"{status_icon} [{event['timestamp']}] "
              f"{event_type} | {sender}→{receiver} | {status}")
    
    def show_audit_trail(self):
        print(f"\n{'='*55}")
        print(f"📋 AUDIT TRAIL")
        print(f"{'='*55}")
        print(f"Total events: {len(self.logs)}\n")
        
        for log in self.logs:
            icon = "✅" if log['status'] == "SUCCESS" else "❌"
            print(f"{icon} {log['timestamp']}")
            print(f"   Event:    {log['event']}")
            print(f"   Parties:  {log['sender']} → {log['receiver']}")
            print(f"   Status:   {log['status']}")
            if log['details']:
                print(f"   Details:  {log['details']}")
            print()

# Initialize logger
logger = AuditLogger()

# Simulate full conversation with logging
print("🔐 SECURECHAT — FULL CONVERSATION LOG\n")

# Message 1 — Alice to Bob (success)
logger.log_event("KEY_EXCHANGE",  "Alice", "Bob", "SUCCESS",
                 "RSA-2048 key exchange completed")

enc1 = encrypt_message_v2(alice_private, bob_public,
                           "Hey Bob! Can you hear me?", "Alice")
result1 = decrypt_message_v2(bob_private, alice_public, enc1, "Alice")
logger.log_event("MESSAGE_SENT", "Alice", "Bob", "SUCCESS",
                 f"Message delivered and verified")

# Message 2 — Bob to Alice (success)
enc2 = encrypt_message_v2(bob_private, alice_public,
                           "Yes Alice! Loud and clear 🔐", "Bob")
result2 = decrypt_message_v2(alice_private, bob_public, enc2, "Bob")
logger.log_event("MESSAGE_SENT", "Bob", "Alice", "SUCCESS",
                 "Message delivered and verified")

# Message 3 — Attack attempt (failure)
tampered = json.loads(enc1)
tampered['sender'] = "Attacker"
try:
    decrypt_message_v2(bob_private, alice_public,
                      json.dumps(tampered), "Alice")
except:
    pass
logger.log_event("ATTACK_DETECTED", "Attacker", "Bob", "BLOCKED",
                 "Sender spoofing attempt rejected")

# Show full audit trail
logger.show_audit_trail()

🔐 SECURECHAT — FULL CONVERSATION LOG

✅ [2026-07-29T12:15:42.550338] KEY_EXCHANGE | Alice→Bob | SUCCESS
📨 MESSAGE RECEIVED
From:      Alice
Timestamp: 2026-07-29T12:15:42.551014

✅ Session key decrypted
✅ Message decrypted: Hey Bob! Can you hear me?
✅ Signature verified — sender authenticated!
✅ Hash verified — integrity confirmed!
✅ [2026-07-29T12:15:42.555488] MESSAGE_SENT | Alice→Bob | SUCCESS
📨 MESSAGE RECEIVED
From:      Bob
Timestamp: 2026-07-29T12:15:42.555895

✅ Session key decrypted
✅ Message decrypted: Yes Alice! Loud and clear 🔐
✅ Signature verified — sender authenticated!
✅ Hash verified — integrity confirmed!
✅ [2026-07-29T12:15:42.558433] MESSAGE_SENT | Bob→Alice | SUCCESS
📨 MESSAGE RECEIVED
From:      Attacker
Timestamp: 2026-07-29T12:15:42.551014
❌ SENDER MISMATCH — Expected Alice, got Attacker
❌ [2026-07-29T12:15:42.558743] ATTACK_DETECTED | Attacker→Bob | BLOCKED

📋 AUDIT TRAIL
Total events: 4

✅ 2026-07-29T12:15:42.550338
   Event:    KEY_EXCHANGE
   Parties:  Alice 